In [13]:
import os
from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types

load_dotenv(find_dotenv())

# Reads GEMINI_API_KEY from your .env file (or you can pass api_key=... explicitly)
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
if os.environ.get("GEMINI_API_KEY"):
    print("Both API Key variable is exists")
else:
    raise ValueError("GEMINI_API_KEY not found")
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from pydantic import BaseModel
llm_genai=ChatGoogleGenerativeAI(model="gemini-3.5-flash",temperature=0.7)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both API Key variable is exists


In [14]:
from typing import Literal
class llm_schema(BaseModel):
    movie_summary_flag : Literal["positive","negative"]
llm_structured_output =llm_genai.with_structured_output(llm_schema)

# Chains with Conditional Chains

In [15]:
#First Chain
#Task -1 [Prompt]
prompt_template=ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant"),
    ("human","{input}")
])

In [16]:
#Task -2 [llm]
llm_genai=ChatGoogleGenerativeAI(model="gemini-3.5-flash",temperature=0.7)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [17]:
#Task -3 [Custom Runnable]
from langchain_core. runnables import RunnableLambda
def pydantic_json(input:llm_schema)->str:
    return input.model_dump()['movie_summary_flag']
pydantic_json_lambda = RunnableLambda(pydantic_json)

# Parallel Chain 1

In [18]:
#Task -1[prompt]
linkedin_prompt=ChatPromptTemplate.from_messages([
    ("system","You are a post generator"),
    ("human","Create a post for the following text for linkedin:{text}")
])

#Task -2 [llm]

llm_genai=ChatGoogleGenerativeAI(model="gemini-3.5-flash",temperature=0.7)

#Task -3[Strout parser]

str_parser=StrOutputParser()

chain_linkedin=linkedin_prompt | llm_genai | str_parser

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


# Parallel Chain 2

In [19]:
from langchain_core.runnables import RunnableSequence,RunnableLambda,RunnableParallel

In [20]:
def insta_chain(text:dict):
    text=text["text"]
    
    #Task -1[prompt]
    insta_prompt=ChatPromptTemplate.from_messages([
        ("system","You are a post generator"),
        ("human","Create a post for the following text for instagram:{text}")
    ])

    #Task -2 [llm]

    llm_genai=ChatGoogleGenerativeAI(model="gemini-3.5-flash",temperature=0.7)

    #Task -3[Strout parser]

    str_parser=StrOutputParser()

    chain_insta=insta_prompt | llm_genai | str_parser
    result=chain_insta.invoke(text)
    return result
insta_chain_runnable=RunnableLambda(insta_chain)

# Final Orchestration

In [11]:
from langchain_core.runnables import RunnableBranch
conditional_chain=RunnableBranch(
    (lambda x: "positive" in x,chain_linkedin),
    insta_chain_runnable
)
final_orchestrator=prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [22]:
final_orchestrator.invoke({"input":"I love this Pushpa movie"})

"Since you didn't include the specific text, I have created **three distinct, positive LinkedIn post templates** depending on what you want to share. \n\nChoose the one that best fits your message and fill in the bracketed details `[like this]`:\n\n---\n\n### Option 1: Celebrating a Win / Milestone (Upbeat & Grateful)\n\n🚀 **Excited to share a major milestone today!** \n\nOver the past few months, our team has been hard at work on `[Project Name / Personal Goal]`, and today we officially `[achieved X goal / launched Y product / reached Z milestone]`!\n\nSuccess is never a solo journey. I want to give a massive shoutout to `[Name/Team]` for their dedication, late nights, and endless creativity. \n\nA few takeaways from this experience:\n1️⃣ **Persistence pays off:** Big goals are just a series of small, consistent steps.\n2️⃣ **Collaboration is everything:** Great ideas get better when shared.\n3️⃣ **Keep learning:** The biggest challenges brought our best growth.\n\nLooking forward to 